# Stage A / NB 04 — Preprocessing and lung-localization cache

Protocol reference: Section 5 (preprocessing and view definitions), Section 9 Stage A NB 04,
experiment family E4. This notebook produces the artifact that agent A1 contributes to the
framework and that every anatomy-aware arm consumes.

## What it does

Runs the already-trained rank-32 anatomy/pathology bbox LoRA **once** over every internal and
external image, and caches four views per image:

| view | definition | consumed by |
| --- | --- | --- |
| V0 | original image, unmodified | E4a, all monolithic arms |
| V1 | thorax crop from the union of both lung boxes, 4% margin | E4b |
| V2L | patient-left lung, orientation-preserving mask on the original canvas | E4d |
| V2R | patient-right lung, same construction | E4d |

The masking construction is taken verbatim from the tested
`anatomy_aware_mrale_medgemma15_5fold` notebook: the canvas size is preserved and everything
outside the lung box is blacked out. It is **not** a resized crop, because resizing a single
lung destroys the PA laterality cue and the relative lung-size cue that mRALE extent
judgement depends on.

Running the localizer once here rather than inside each experiment is what keeps the Stage C
compute budget finite, and it also guarantees that every arm sees identical boxes — so E4c
versus E4d measures the effect of the *views*, not of localizer sampling noise.

## Outputs (under `stage_A/nb04_localization/`)
- `anatomy_localizations.jsonl`   checkpointed, one row per image, resumable
- `views/{v1_thorax,v2_left,v2_right}/*.png`
- `view_index.csv`                the lookup table every later notebook joins on
- `localization_metrics.csv`, `localization_metrics.json`
- `localization_qc_panel_*.png`
- `run_config.json`, `gate_nb04.json`

## Gate
- Box validity rate on internal images >= 0.95 (protocol G1).
- Fallback rate recorded and reported alongside every model result that uses this cache.
- Visual QC panel reviewed and signed off before Stage B consumes the cache.

## 1. Imports

In [ ]:
import csv
import gc
import json
import math
import os
import random
import re
import sys
import time
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw
from peft import PeftModel
from transformers import AutoModelForImageTextToText, AutoProcessor

Image.MAX_IMAGE_PIXELS = None

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

## 2. Configuration

Geometry constants match the tested anatomy-aware notebook exactly (`COORDINATE_SCALE=1000`,
`LUNG_BOX_MARGIN_FRACTION=0.04`). Changing them here would silently change the meaning of
every E4 result, so they are asserted rather than assumed.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

FALLBACK_STAGE_A_DIR = Path("/data/liangz2/openi/midrc/tetci_resubmit/stage_A")
PATHS_JSON_CANDIDATES = [
    FALLBACK_STAGE_A_DIR / "nb00_environment" / "stage_a_paths.json",
    Path.cwd() / "stage_a_paths.json",
]
stage_paths = None
for candidate in PATHS_JSON_CANDIDATES:
    if candidate.is_file():
        stage_paths = json.loads(candidate.read_text(encoding="utf-8"))
        print("Loaded path contract from:", candidate)
        break
if stage_paths is None:
    raise FileNotFoundError("stage_a_paths.json not found. Run NB 00 first.")

NB01_DIR = Path(stage_paths["nb_output_dirs"]["nb01_inventory"])
NB02_DIR = Path(stage_paths["nb_output_dirs"]["nb02_folds"])
NB03_DIR = Path(stage_paths["nb_output_dirs"]["nb03_external"])
NB04_DIR = Path(stage_paths["nb_output_dirs"]["nb04_localization"])
VIEW_DIR = NB04_DIR / "views"
for directory in [NB04_DIR, VIEW_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

BASE_MODEL_ID = "google/medgemma-1.5-4b-it"
BASE_MODEL_REVISION = stage_paths.get("model_revisions", {}).get(BASE_MODEL_ID)
ANATOMY_ADAPTER_DIR = Path(stage_paths["legacy"]["bbox_adapter_dir"])

# Geometry, unchanged from the tested notebook.
COORDINATE_SCALE = 1000
LUNG_BOX_MARGIN_FRACTION = 0.04
ALLOW_HEURISTIC_BOX_FALLBACK = True
MAX_ANATOMY_NEW_TOKENS = 1000

# Which cohorts to localize.
INCLUDE_INTERNAL = True
INCLUDE_EXTERNAL = True
MAX_IMAGES = None            # Set to 16 for a smoke test, then None for the full run.
WRITE_V1_THORAX = True
WRITE_V2_REGIONAL = True
OVERWRITE_EXISTING_VIEWS = False

# Optional intensity normalization arm for E9c. OFF by default: the primary pipeline keeps
# original pixels so that the cache is faithful to what the tested notebook produced.
APPLY_INTENSITY_NORMALIZATION = False
CLIP_PERCENTILES = (0.5, 99.5)
APPLY_CLAHE = False

QC_PANEL_SAMPLE_SIZE = 40
QC_PANEL_COLUMNS = 5

assert COORDINATE_SCALE == 1000, "E4 results are only comparable at the trained scale."
assert abs(LUNG_BOX_MARGIN_FRACTION - 0.04) < 1e-9, "Margin must match the tested pipeline."
assert ANATOMY_ADAPTER_DIR.joinpath("adapter_config.json").is_file(), ANATOMY_ADAPTER_DIR
if BASE_MODEL_REVISION is None:
    print("WARNING: no pinned revision for", BASE_MODEL_ID,
          "- NB 00 did not record one. Proceeding without revision pinning.")

print("Base model:", BASE_MODEL_ID, "revision:", BASE_MODEL_REVISION)
print("Anatomy adapter:", ANATOMY_ADAPTER_DIR)
print("Output:", NB04_DIR)

## 3. Assemble the work list

Internal images come from `midrc_manifest.csv` (primary cohort only). External images come
from every `*_manifest.csv` NB 03 prepared. Each entry gets a globally unique `image_key`
of the form `cohort::filename`, which becomes the join key for Stages B-D.

In [ ]:
def read_jsonl(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
    return rows


work = []

if INCLUDE_INTERNAL:
    manifest_csv = NB01_DIR / "midrc_manifest.csv"
    if not manifest_csv.is_file():
        raise FileNotFoundError(f"{manifest_csv} not found. Run NB 01 first.")
    internal = pd.read_csv(manifest_csv)
    internal = internal[(internal["in_primary_cohort"] == True) &
                        (internal["image_status"] == "OK")]
    fold_csv = NB02_DIR / "fold_definitions" / "midrc_folds_v2.csv"
    fold_by_filename = {}
    if fold_csv.is_file():
        folds = pd.read_csv(fold_csv)
        fold_by_filename = dict(zip(folds["filename"].astype(str), folds["fold"]))
    else:
        print(f"NOTE: {fold_csv} not found; held_out_fold will be null. Run NB 02 to "
              "populate it.")
    for _, row in internal.iterrows():
        work.append({
            "image_key": f"MIDRC::{row['filename']}",
            "cohort": "MIDRC",
            "subcohort": "MIDRC",
            "filename": str(row["filename"]),
            "image_path": str(row["image_path"]),
            "held_out_fold": fold_by_filename.get(str(row["filename"])),
            "width": int(row["width"]) if pd.notna(row.get("width")) else None,
            "height": int(row["height"]) if pd.notna(row.get("height")) else None,
        })
    print(f"Internal images queued: {len(work):,}")

if INCLUDE_EXTERNAL:
    external_dir = NB03_DIR / "external_manifests"
    if not external_dir.is_dir():
        print(f"NOTE: {external_dir} not found; skipping external cohorts. Run NB 03 to "
              "enable the E9 anatomy-aware arm.")
    else:
        before = len(work)
        for manifest_path in sorted(external_dir.glob("*_manifest.csv")):
            frame = pd.read_csv(manifest_path)
            if "status" in frame.columns:
                frame = frame[frame["status"] == "OK"]
            for _, row in frame.iterrows():
                cohort = str(row.get("cohort", manifest_path.stem.split("_")[0]))
                work.append({
                    "image_key": f"{cohort}::{row['filename']}",
                    "cohort": cohort,
                    "subcohort": str(row.get("subcohort", cohort)),
                    "filename": str(row["filename"]),
                    "image_path": str(row["image_path"]),
                    "held_out_fold": None,
                    "width": int(row["width"]) if pd.notna(row.get("width")) else None,
                    "height": int(row["height"]) if pd.notna(row.get("height")) else None,
                })
        print(f"External images queued: {len(work) - before:,}")

keys = [item["image_key"] for item in work]
duplicates = [key for key, count in Counter(keys).items() if count > 1]
if duplicates:
    raise ValueError(f"image_key is not unique: {duplicates[:10]}")

work.sort(key=lambda item: item["image_key"])
if MAX_IMAGES is not None:
    work = work[:MAX_IMAGES]
    print(f"SMOKE TEST: truncated to {len(work)} images.")

print()
print(f"Total work list: {len(work):,} images")
print("By cohort:", dict(Counter(item["cohort"] for item in work)))
if not work:
    raise RuntimeError("Nothing to localize.")

## 4. Prompts, model loading, and generation

Prompt text and generation settings are copied from the tested anatomy-aware notebook without
modification. The adapter was trained against these exact strings, so paraphrasing them would
degrade box quality for no benefit.

In [ ]:
ANATOMY_SYSTEM_PROMPT = (
    "You are a chest radiograph localization assistant. Identify the annotated thoracic anatomy. "
    "Return only valid JSON with coordinate_system and anatomy. Each anatomy item must contain "
    "region, laterality, and bbox. bbox is [x1,y1,x2,y2] in a 0-1000 coordinate system relative "
    "to the full image, with the origin at the upper-left."
)
ANATOMY_USER_PROMPT = "Localize the annotated thoracic anatomy in this frontal chest X-ray."


def multimodal_messages(system_prompt, user_prompt):
    return [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": user_prompt},
        ]},
    ]


processor_kwargs = {}
if BASE_MODEL_REVISION:
    processor_kwargs["revision"] = BASE_MODEL_REVISION
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, **processor_kwargs)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"


def load_adapter_model(adapter_dir):
    kwargs = {
        "torch_dtype": torch.bfloat16,
        "device_map": "auto",
        "low_cpu_mem_usage": True,
    }
    if BASE_MODEL_REVISION:
        kwargs["revision"] = BASE_MODEL_REVISION
    base = AutoModelForImageTextToText.from_pretrained(BASE_MODEL_ID, **kwargs)
    base.config.use_cache = True
    model = PeftModel.from_pretrained(base, str(adapter_dir), is_trainable=False)
    model.eval()
    return model


def model_input_device(model):
    for parameter in model.parameters():
        if parameter.device.type not in {"meta", "cpu"}:
            return parameter.device
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def extract_json_object(text):
    cleaned = re.sub(r"^```(?:json)?\s*", "", text.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    start = cleaned.find("{")
    if start < 0:
        raise ValueError("No JSON object found")
    obj, _ = json.JSONDecoder().raw_decode(cleaned[start:])
    if not isinstance(obj, dict):
        raise ValueError("Prediction is not a JSON object")
    return obj


def safe_parse(text):
    try:
        return extract_json_object(text), None
    except Exception as error:
        return None, str(error)


@torch.inference_mode()
def generate(model, image_path, system_prompt, user_prompt, max_new_tokens):
    prompt = processor.apply_chat_template(
        multimodal_messages(system_prompt, user_prompt),
        add_generation_prompt=True,
        tokenize=False,
    )
    with Image.open(image_path) as image_file:
        image = image_file.convert("RGB")
        inputs = processor(text=prompt, images=image, return_tensors="pt")
    device = model_input_device(model)
    moved = {
        key: value.to(device=device, dtype=torch.bfloat16)
        if value.is_floating_point() else value.to(device)
        for key, value in inputs.items()
    }
    prompt_length = moved["input_ids"].shape[-1]
    output_ids = model.generate(
        **moved,
        do_sample=False,
        max_new_tokens=max_new_tokens,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
        use_cache=True,
    )
    return processor.decode(output_ids[0, prompt_length:], skip_special_tokens=True).strip()


def load_jsonl_by_key(path, key_fields):
    rows = {}
    if not Path(path).is_file():
        return rows
    with Path(path).open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
                rows[tuple(str(row[field]) for field in key_fields)] = row
            except Exception as error:
                print(f"Ignoring malformed {Path(path).name} line {line_number}: {error}")
    return rows


def append_jsonl(path, row):
    with Path(path).open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())

## 5. Box extraction, geometry, and view construction

`extract_lung_boxes`, `expand_box`, and the masked-view construction are the tested
implementations. Two additions:

- `v1_thorax_view` builds the union-box thorax crop needed by E4b, which the original
  notebook did not produce.
- `box_geometry_checks` records whether each prediction is anatomically plausible on a PA
  display (patient-left lung appears on the image's right). A localizer that swaps
  laterality would otherwise silently invert the per-lung scores, and E4 would report the
  wrong conclusion for the right reason.

In [ ]:
def valid_box(box):
    try:
        x1, y1, x2, y2 = [float(value) for value in box]
        return 0 <= x1 < x2 <= COORDINATE_SCALE and 0 <= y1 < y2 <= COORDINATE_SCALE
    except (TypeError, ValueError):
        return False


def normalized_laterality(item):
    laterality = str(item.get("laterality", "")).strip().lower()
    region = str(item.get("region", "")).strip().lower()
    if laterality in {"left", "right"}:
        return laterality
    if "left" in region:
        return "left"
    if "right" in region:
        return "right"
    return None


def union_boxes(boxes):
    return [
        min(box[0] for box in boxes), min(box[1] for box in boxes),
        max(box[2] for box in boxes), max(box[3] for box in boxes),
    ]


HEURISTIC_BOXES = {
    # On a standard PA display, patient left is on the image's right.
    "left": [450, 80, 1000, 970],
    "right": [0, 80, 550, 970],
}


def extract_lung_boxes(parsed):
    candidates = {"left": [], "right": []}
    anatomy = parsed.get("anatomy", []) if isinstance(parsed, dict) else []
    for item in anatomy:
        if not isinstance(item, dict) or not valid_box(item.get("bbox")):
            continue
        region = str(item.get("region", "")).lower()
        if "lung" not in region or "hil" in region:
            continue
        laterality = normalized_laterality(item)
        if laterality:
            candidates[laterality].append([float(value) for value in item["bbox"]])
    boxes = {
        side: union_boxes(side_boxes) if side_boxes else None
        for side, side_boxes in candidates.items()
    }
    sources = {side: "anatomy_lora" if boxes[side] is not None else None for side in boxes}
    for side in ["left", "right"]:
        if boxes[side] is None:
            if not ALLOW_HEURISTIC_BOX_FALLBACK:
                raise ValueError(f"No {side} lung box returned by the anatomy model")
            boxes[side] = list(HEURISTIC_BOXES[side])
            sources[side] = "heuristic_fallback"
    return boxes, sources


def expand_box(box, margin_fraction=LUNG_BOX_MARGIN_FRACTION):
    x1, y1, x2, y2 = box
    width, height = x2 - x1, y2 - y1
    return [
        max(0, x1 - margin_fraction * width),
        max(0, y1 - margin_fraction * height),
        min(COORDINATE_SCALE, x2 + margin_fraction * width),
        min(COORDINATE_SCALE, y2 + margin_fraction * height),
    ]


def to_pixel_box(normalized_box, width, height):
    x1, y1, x2, y2 = normalized_box
    return (
        max(0, round(x1 * width / COORDINATE_SCALE)),
        max(0, round(y1 * height / COORDINATE_SCALE)),
        min(width, round(x2 * width / COORDINATE_SCALE)),
        min(height, round(y2 * height / COORDINATE_SCALE)),
    )


def normalize_intensity(image):
    # E9c arm only. Percentile clip then min-max to the full 8-bit range.
    array = np.asarray(image.convert("L"), dtype=np.float32)
    low, high = np.percentile(array, CLIP_PERCENTILES)
    if high <= low:
        return image
    array = np.clip(array, low, high)
    array = (array - low) / (high - low) * 255.0
    return Image.fromarray(array.astype(np.uint8)).convert("RGB")


def prepare_source_image(image_path):
    with Image.open(image_path) as handle:
        image = handle.convert("RGB")
    if APPLY_INTENSITY_NORMALIZATION:
        image = normalize_intensity(image)
    return image


def masked_view(image, normalized_box, output_path):
    # Preserves the original canvas; everything outside the box is black. NOT a resized crop.
    width, height = image.size
    pixel_box = to_pixel_box(expand_box(normalized_box), width, height)
    canvas = Image.new("RGB", image.size, (0, 0, 0))
    canvas.paste(image.crop(pixel_box), pixel_box[:2])
    output_path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(output_path)
    return str(output_path)


def v1_thorax_view(image, left_box, right_box, output_path):
    # E4b: a genuine crop of the thorax, from the union of both lung boxes.
    width, height = image.size
    union = union_boxes([left_box, right_box])
    pixel_box = to_pixel_box(expand_box(union), width, height)
    if pixel_box[2] <= pixel_box[0] or pixel_box[3] <= pixel_box[1]:
        return None
    output_path.parent.mkdir(parents=True, exist_ok=True)
    image.crop(pixel_box).save(output_path)
    return str(output_path)


def box_geometry_checks(boxes, sources):
    left, right = boxes["left"], boxes["right"]
    left_centre = (left[0] + left[2]) / 2.0
    right_centre = (right[0] + right[2]) / 2.0
    left_area = (left[2] - left[0]) * (left[3] - left[1]) / COORDINATE_SCALE ** 2
    right_area = (right[2] - right[0]) * (right[3] - right[1]) / COORDINATE_SCALE ** 2
    overlap_width = max(0.0, min(left[2], right[2]) - max(left[0], right[0]))
    overlap_height = max(0.0, min(left[3], right[3]) - max(left[1], right[1]))
    overlap_area = overlap_width * overlap_height / COORDINATE_SCALE ** 2
    smaller = min(left_area, right_area)
    return {
        "left_centre_x": round(left_centre, 2),
        "right_centre_x": round(right_centre, 2),
        # On a PA display the patient's left lung must sit to the image right of the right lung.
        "laterality_plausible": bool(left_centre > right_centre),
        "left_area_fraction": round(left_area, 4),
        "right_area_fraction": round(right_area, 4),
        "area_ratio": round(left_area / right_area, 3) if right_area > 0 else None,
        "box_overlap_fraction": round(overlap_area / smaller, 3) if smaller > 0 else None,
        "any_fallback": any(sources[side] != "anatomy_lora" for side in ["left", "right"]),
    }

## 6. Run the localizer

Checkpointed per image: interrupting and re-running resumes where it stopped. The model is
loaded once and released at the end. Expect roughly 1 s per image on an A100, so a full
internal-plus-external pass is a few GPU-hours.

In [ ]:
LOCALIZATION_PATH = NB04_DIR / "anatomy_localizations.jsonl"
localizations = load_jsonl_by_key(LOCALIZATION_PATH, ["image_key"])
print(f"Resuming with {len(localizations):,} already-localized images.")

pending = [item for item in work if (item["image_key"],) not in localizations]
print(f"Pending: {len(pending):,}")

if pending:
    anatomy_model = load_adapter_model(ANATOMY_ADAPTER_DIR)
    started = time.perf_counter()
    for index, item in enumerate(pending, start=1):
        image_started = time.perf_counter()
        raw = generate(
            anatomy_model, item["image_path"], ANATOMY_SYSTEM_PROMPT,
            ANATOMY_USER_PROMPT, MAX_ANATOMY_NEW_TOKENS,
        )
        parsed, parse_error = safe_parse(raw)
        boxes, sources = extract_lung_boxes(parsed or {})
        geometry = box_geometry_checks(boxes, sources)

        image = prepare_source_image(item["image_path"])
        width, height = image.size
        view_paths = {}
        if WRITE_V2_REGIONAL:
            for side, key in [("left", "v2_left"), ("right", "v2_right")]:
                target = VIEW_DIR / key / f"{item['image_key'].replace('::', '__')}.png"
                if target.is_file() and not OVERWRITE_EXISTING_VIEWS:
                    view_paths[key] = str(target)
                else:
                    view_paths[key] = masked_view(image, boxes[side], target)
        if WRITE_V1_THORAX:
            target = VIEW_DIR / "v1_thorax" / f"{item['image_key'].replace('::', '__')}.png"
            if target.is_file() and not OVERWRITE_EXISTING_VIEWS:
                view_paths["v1_thorax"] = str(target)
            else:
                view_paths["v1_thorax"] = v1_thorax_view(
                    image, boxes["left"], boxes["right"], target)
        image.close()

        n_anatomy_items = len(parsed.get("anatomy", [])) if isinstance(parsed, dict) else 0
        row = {
            "image_key": item["image_key"],
            "cohort": item["cohort"],
            "subcohort": item["subcohort"],
            "filename": item["filename"],
            "image_path": item["image_path"],
            "held_out_fold": item["held_out_fold"],
            "source_width": width,
            "source_height": height,
            "raw_anatomy": raw,
            "parsed_anatomy": parsed,
            "parse_error": parse_error,
            "n_anatomy_items": n_anatomy_items,
            "left_box": boxes["left"],
            "right_box": boxes["right"],
            "left_box_source": sources["left"],
            "right_box_source": sources["right"],
            "v0_image": item["image_path"],
            "v1_thorax_image": view_paths.get("v1_thorax"),
            "v2_left_image": view_paths.get("v2_left"),
            "v2_right_image": view_paths.get("v2_right"),
            "seconds": round(time.perf_counter() - image_started, 3),
            **geometry,
        }
        append_jsonl(LOCALIZATION_PATH, row)
        localizations[(item["image_key"],)] = row

        if index % 50 == 0 or index == len(pending):
            elapsed = time.perf_counter() - started
            rate = index / elapsed
            remaining = (len(pending) - index) / rate if rate > 0 else float("nan")
            print(f"  [{index}/{len(pending)}] {rate:.2f} img/s, "
                  f"~{remaining / 60:.1f} min remaining")

    del anatomy_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("Nothing pending; using the existing cache.")

print()
print(f"Cache size: {len(localizations):,} images")

## 7. View index and localization metrics

`view_index.csv` is the contract for Stages B-D: one row per image, with the absolute path of
every view. Downstream notebooks join on `image_key` and never rebuild a view.

In [ ]:
INDEX_COLUMNS = [
    "image_key", "cohort", "subcohort", "filename", "held_out_fold",
    "source_width", "source_height",
    "v0_image", "v1_thorax_image", "v2_left_image", "v2_right_image",
    "left_box", "right_box", "left_box_source", "right_box_source",
    "n_anatomy_items", "parse_error",
    "laterality_plausible", "left_area_fraction", "right_area_fraction",
    "area_ratio", "box_overlap_fraction", "any_fallback", "seconds",
]

index_rows = []
for item in work:
    row = localizations.get((item["image_key"],))
    if row is None:
        continue
    flat = {column: row.get(column) for column in INDEX_COLUMNS}
    for side in ["left_box", "right_box"]:
        if isinstance(flat[side], list):
            flat[side] = json.dumps([round(float(value), 2) for value in flat[side]])
    index_rows.append(flat)

view_index = pd.DataFrame(index_rows, columns=INDEX_COLUMNS)
view_index.to_csv(NB04_DIR / "view_index.csv", index=False)
print(f"view_index.csv: {len(view_index):,} rows")
print()
print(view_index[["image_key", "cohort", "left_box_source", "right_box_source",
                  "laterality_plausible"]].head(8).to_string(index=False))

In [ ]:
def cohort_metrics(subset, label):
    total = len(subset)
    if total == 0:
        return None
    lora_left = int((subset["left_box_source"] == "anatomy_lora").sum())
    lora_right = int((subset["right_box_source"] == "anatomy_lora").sum())
    both_lora = int(((subset["left_box_source"] == "anatomy_lora") &
                     (subset["right_box_source"] == "anatomy_lora")).sum())
    parse_ok = int(subset["parse_error"].isna().sum())
    return OrderedDict([
        ("cohort", label),
        ("n_images", total),
        ("json_parse_rate", round(parse_ok / total, 4)),
        ("left_box_validity_rate", round(lora_left / total, 4)),
        ("right_box_validity_rate", round(lora_right / total, 4)),
        ("both_boxes_valid_rate", round(both_lora / total, 4)),
        ("fallback_rate", round(1 - both_lora / total, 4)),
        ("laterality_plausible_rate", round(
            float(subset["laterality_plausible"].mean()), 4)),
        ("mean_left_area_fraction", round(float(subset["left_area_fraction"].mean()), 4)),
        ("mean_right_area_fraction", round(float(subset["right_area_fraction"].mean()), 4)),
        ("median_area_ratio", round(float(subset["area_ratio"].median()), 3)
         if subset["area_ratio"].notna().any() else None),
        ("mean_box_overlap_fraction", round(
            float(subset["box_overlap_fraction"].mean()), 4)
         if subset["box_overlap_fraction"].notna().any() else None),
        ("mean_anatomy_items", round(float(subset["n_anatomy_items"].mean()), 2)),
        ("median_seconds_per_image", round(float(subset["seconds"].median()), 3)
         if subset["seconds"].notna().any() else None),
    ])


metric_rows = [cohort_metrics(view_index, "ALL")]
for cohort in sorted(view_index["cohort"].dropna().unique()):
    metric_rows.append(cohort_metrics(view_index[view_index["cohort"] == cohort], cohort))
for subcohort in sorted(view_index["subcohort"].dropna().unique()):
    if subcohort not in set(view_index["cohort"].dropna().unique()):
        metric_rows.append(cohort_metrics(
            view_index[view_index["subcohort"] == subcohort], f"  {subcohort}"))

metrics = pd.DataFrame([row for row in metric_rows if row is not None])
metrics.to_csv(NB04_DIR / "localization_metrics.csv", index=False)
with (NB04_DIR / "localization_metrics.json").open("w", encoding="utf-8") as handle:
    json.dump(metrics.to_dict(orient="records"), handle, indent=2, default=str)

print(metrics.to_string(index=False))
print()
print("Report fallback_rate and laterality_plausible_rate alongside every model result that "
      "consumes this cache. Protocol Section 6, E4: the anatomy-aware arm's credibility "
      "depends on the localizer's measured quality, not on its assumed quality.")

implausible = view_index[view_index["laterality_plausible"] == False]
if len(implausible):
    print()
    print(f"WARNING: {len(implausible)} images have implausible laterality (patient-left box "
          "is not to the image right of the patient-right box). Inspect these before "
          "trusting per-lung scores.")
    detail_columns = [column for column in
                      ["image_key", "cohort", "left_box", "right_box",
                       "left_box_source", "right_box_source"]
                      if column in implausible.columns]
    implausible[detail_columns].to_csv(
        NB04_DIR / "implausible_laterality.csv", index=False)
    print(implausible[detail_columns].head(10).to_string(index=False))

## 8. Visual QC panel

Stratified sample: fallback cases and implausible-laterality cases are over-sampled on
purpose, because a random sample of a 95%-clean cache mostly shows successes and teaches
nothing. Review these before Stage B runs.

In [ ]:
def draw_overlay(row, target_height=420):
    with Image.open(row["v0_image"]) as handle:
        image = handle.convert("RGB")
    draw = ImageDraw.Draw(image)
    width, height = image.size
    for side, colour in [("left_box", "#00D4FF"), ("right_box", "#FF3B30")]:
        box = json.loads(row[side]) if isinstance(row[side], str) else row[side]
        pixel_box = to_pixel_box(box, width, height)
        draw.rectangle(pixel_box, outline=colour, width=max(2, width // 250))
    scale = target_height / height
    return image.resize((max(1, int(width * scale)), target_height), Image.Resampling.LANCZOS)


rng = random.Random(SEED)
candidates = view_index.to_dict(orient="records")
priority = [row for row in candidates
            if row.get("any_fallback") or row.get("laterality_plausible") is False]
ordinary = [row for row in candidates if row not in priority]
rng.shuffle(priority)
rng.shuffle(ordinary)
sample = (priority[:QC_PANEL_SAMPLE_SIZE // 2] +
          ordinary[:QC_PANEL_SAMPLE_SIZE - len(priority[:QC_PANEL_SAMPLE_SIZE // 2])])
print(f"QC panel: {len(sample)} images "
      f"({len(priority[:QC_PANEL_SAMPLE_SIZE // 2])} fallback/implausible, "
      f"{len(sample) - len(priority[:QC_PANEL_SAMPLE_SIZE // 2])} ordinary)")

panel_paths = []
if sample:
    tiles = []
    for row in sample:
        try:
            tiles.append((row, draw_overlay(row)))
        except Exception as exc:
            print(f"  could not render {row['image_key']}: {exc}")
    if tiles:
        columns = QC_PANEL_COLUMNS
        tile_width = max(tile.width for _, tile in tiles)
        tile_height = max(tile.height for _, tile in tiles)
        rows_needed = math.ceil(len(tiles) / columns)
        panel = Image.new("RGB", (columns * tile_width, rows_needed * tile_height),
                          (16, 16, 16))
        for position, (row, tile) in enumerate(tiles):
            x = (position % columns) * tile_width
            y = (position // columns) * tile_height
            panel.paste(tile, (x, y))
            draw = ImageDraw.Draw(panel)
            label = f"{row['image_key'][:28]} {row['left_box_source'][:4]}/{row['right_box_source'][:4]}"
            draw.text((x + 4, y + 4), label, fill="#FFD400")
        panel_path = NB04_DIR / "localization_qc_panel_01.png"
        panel.save(panel_path)
        panel_paths.append(str(panel_path))
        print("Wrote:", panel_path)

    # A few single-image detail figures, useful directly as manuscript Fig. 2 material.
    for position, row in enumerate(sample[:3]):
        try:
            views = [("V0 overlay", draw_overlay(row))]
            for label, key in [("V1 thorax", "v1_thorax_image"),
                               ("V2 left", "v2_left_image"),
                               ("V2 right", "v2_right_image")]:
                path = row.get(key)
                if path and Path(path).is_file():
                    with Image.open(path) as handle:
                        view = handle.convert("RGB")
                    scale = 420 / view.height
                    views.append((label, view.resize(
                        (max(1, int(view.width * scale)), 420), Image.Resampling.LANCZOS)))
            strip_width = sum(view.width for _, view in views)
            strip = Image.new("RGB", (strip_width, 420), (16, 16, 16))
            offset = 0
            draw = ImageDraw.Draw(strip)
            for label, view in views:
                strip.paste(view, (offset, 0))
                draw.text((offset + 4, 4), label, fill="#FFD400")
                offset += view.width
            detail_path = NB04_DIR / f"localization_views_example_{position}.png"
            strip.save(detail_path)
            panel_paths.append(str(detail_path))
            print("Wrote:", detail_path)
        except Exception as exc:
            print(f"  could not render view strip for {row['image_key']}: {exc}")

## 9. Run configuration and gate

In [ ]:
run_config = {
    "written_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "04_preprocessing_and_lung_localization_cache.ipynb",
    "seed": SEED,
    "base_model_id": BASE_MODEL_ID,
    "base_model_revision": BASE_MODEL_REVISION,
    "anatomy_adapter_dir": str(ANATOMY_ADAPTER_DIR),
    "geometry": {
        "coordinate_scale": COORDINATE_SCALE,
        "lung_box_margin_fraction": LUNG_BOX_MARGIN_FRACTION,
        "allow_heuristic_box_fallback": ALLOW_HEURISTIC_BOX_FALLBACK,
        "heuristic_boxes": HEURISTIC_BOXES,
        "max_anatomy_new_tokens": MAX_ANATOMY_NEW_TOKENS,
    },
    "preprocessing": {
        "apply_intensity_normalization": APPLY_INTENSITY_NORMALIZATION,
        "clip_percentiles": list(CLIP_PERCENTILES),
        "apply_clahe": APPLY_CLAHE,
        "note": (
            "The primary cache preserves original pixels. Intensity normalization is the "
            "E9c domain-shift arm and is produced by re-running this notebook into a "
            "separate output directory with APPLY_INTENSITY_NORMALIZATION=True."
        ),
    },
    "views": {
        "v0": "original image, unmodified",
        "v1_thorax": "crop of the union lung box with 4% margin",
        "v2_left": "patient-left lung, original canvas, outside masked black",
        "v2_right": "patient-right lung, original canvas, outside masked black",
    },
    "decoding": {"do_sample": False, "deterministic": True},
    "counts": {
        "work_list": len(work),
        "localized": len(view_index),
        "cohorts": {str(key): int(value)
                    for key, value in Counter(view_index["cohort"]).items()},
    },
    "qc_panels": panel_paths,
}
with (NB04_DIR / "run_config.json").open("w", encoding="utf-8") as handle:
    json.dump(run_config, handle, indent=2, default=str)

failures = []
warnings = []

if len(view_index) != len(work):
    failures.append(f"Localized {len(view_index)} of {len(work)} queued images.")

internal_view = view_index[view_index["cohort"] == "MIDRC"]
if len(internal_view):
    both_valid = float(((internal_view["left_box_source"] == "anatomy_lora") &
                        (internal_view["right_box_source"] == "anatomy_lora")).mean())
    print(f"Internal both-boxes-valid rate: {both_valid:.4f}")
    if both_valid < 0.95:
        failures.append(
            f"Internal box validity {both_valid:.4f} is below the protocol G1 threshold of "
            "0.95. Investigate before Stage B: either the adapter is being applied "
            "incorrectly, or the prompt/processor pairing has drifted."
        )
    laterality_rate = float(internal_view["laterality_plausible"].mean())
    if laterality_rate < 0.98:
        failures.append(
            f"Laterality plausible on only {laterality_rate:.4f} of internal images. "
            "Per-lung mRALE scores would be assigned to the wrong lung."
        )
elif INCLUDE_INTERNAL:
    failures.append("No internal images in the cache despite INCLUDE_INTERNAL=True.")

for column in ["v2_left_image", "v2_right_image"]:
    if WRITE_V2_REGIONAL:
        missing = int(view_index[column].isna().sum())
        if missing:
            failures.append(f"{missing} images are missing {column}.")
if WRITE_V1_THORAX:
    missing_v1 = int(view_index["v1_thorax_image"].isna().sum())
    if missing_v1:
        warnings.append(f"{missing_v1} images have no V1 thorax crop (degenerate union box). "
                        "E4b coverage is reduced; report it.")

parse_failures = int(view_index["parse_error"].notna().sum())
if parse_failures:
    warnings.append(f"{parse_failures} images produced unparseable localizer output and fell "
                    "back to the heuristic boxes.")

if MAX_IMAGES is not None:
    warnings.append(f"SMOKE TEST MODE: only {MAX_IMAGES} images processed. Set "
                    "MAX_IMAGES=None and re-run before Stage B.")

if not panel_paths:
    warnings.append("No QC panel was produced. Visual review is a protocol gate, not "
                    "optional.")
else:
    warnings.append("QC panels written. Review them and record sign-off before Stage B "
                    "consumes this cache.")


def report(title, messages):
    print(title)
    if messages:
        for message in messages:
            print("  -", message)
    else:
        print("  none")


print()
report("WARNINGS", warnings)
print()
report("FAILURES", failures)

with (NB04_DIR / "gate_nb04.json").open("w", encoding="utf-8") as handle:
    json.dump({"passed": not failures, "failures": failures, "warnings": warnings},
              handle, indent=2)

assert not failures, f"NB 04 gate failed with {len(failures)} blocking issue(s)."
print()
print("NB 04 gate: PASSED")
print("Stage A complete. Stage B reads view_index.csv and the NB 02 fold definitions.")

## Notes carried forward

- `view_index.csv` is the join table for every anatomy-aware arm. Stage B/C notebooks look up
  `image_key = "<cohort>::<filename>"` and read `v0_image`, `v1_thorax_image`,
  `v2_left_image`, `v2_right_image`. No later notebook re-runs the localizer.
- **Boxes are frozen here on purpose.** E4c versus E4d must differ only in how the views are
  presented to the predictor. If each experiment re-localized, localizer sampling noise would
  contaminate the comparison that the novelty claim rests on.
- **E4f needs ground-truth boxes.** The oracle ceiling arm requires manually or externally
  annotated lung boxes for a subset of images; they are not produced here. Plan for a
  100-200 image annotated subset, and note that without E4f a negative E4 result cannot be
  attributed between "the idea is wrong" and "the localizer is not good enough".
- **E9c** is produced by re-running this notebook with `APPLY_INTENSITY_NORMALIZATION=True`
  into a separate output directory. Do not overwrite the primary cache.
- The `fallback_rate` and `laterality_plausible_rate` columns of `localization_metrics.csv`
  belong in the manuscript next to every anatomy-aware number.